In [14]:
%%writefile main.cpp
#include <iostream>
#include <vector>
#include <omp.h>

#define GRAIN_SIZE 128

// Improved Algorithm with Grain Size
long parallel_sum(long* A, int n, int grain_size) {
    if (n <= grain_size) { // Optimization: Base case uses a serial loop
        long sum = 0;
        for (int i = 0; i < n; i++) sum += A[i];
        return sum;
    } else {
        // Divide and Conquer
        long x = spawn parallel_sum(A, n/2, grain_size);
        long y = parallel_sum(A + n/2, n - n/2, grain_size);
        sync;
        return x + y;
    }
}

    int mid = left + size / 2;
    long long sum1 = 0, sum2 = 0;

    #pragma omp task shared(sum1)
    sum1 = parallel_sum(arr, left, mid);

    #pragma omp task shared(sum2)
    sum2 = parallel_sum(arr, mid, right);

    #pragma omp taskwait

    return sum1 + sum2;
}

int main() {

    int n = 1 << 26;   // (larger for better measurement)
    std::vector<int> arr(n, 1);

    long long result = 0;

    int threads = omp_get_max_threads();
    std::cout << "Threads used: " << threads << std::endl;

    double start = omp_get_wtime();

    #pragma omp parallel
    {
        #pragma omp single
        result = parallel_sum(arr.data(), 0, n);
    }

    double end = omp_get_wtime();

    std::cout << "Sum = " << result << std::endl;
    std::cout << "Execution time = " << end - start << " seconds\n";

    return 0;
}

Overwriting main.cpp


In [16]:
!OMP_NUM_THREADS=1 ./main
!OMP_NUM_THREADS=2 ./main
!OMP_NUM_THREADS=4 ./main
!OMP_NUM_THREADS=8 ./main
!OMP_NUM_THREADS=16 ./main
!OMP_NUM_THREADS=32 ./main
!OMP_NUM_THREADS=64 ./main
!OMP_NUM_THREADS=128 ./main


Threads used: 1
Sum = 67108864
Execution time = 0.351504 seconds
Threads used: 2
Sum = 67108864
Execution time = 0.358363 seconds
Threads used: 4
Sum = 67108864
Execution time = 0.403116 seconds
Threads used: 8
Sum = 67108864
Execution time = 0.402344 seconds
Threads used: 16
Sum = 67108864
Execution time = 0.397407 seconds
Threads used: 32
Sum = 67108864
Execution time = 0.407947 seconds
Threads used: 64
Sum = 67108864
Execution time = 0.429984 seconds
Threads used: 128
Sum = 67108864
Execution time = 0.443909 seconds
